# 13 — GoalScope qualitative demo (paper-style callouts)

Interactive version of `Overleaf-draft/REF/qualitative_demo.png`: source code on the right, italic-quote read-out callouts on the left joined by leader lines to pink anchor dots.

**Interaction:** click a highlighted (scoped) token to pin its read-out — a numbered callout appears, the token is boxed + dotted, and a curve is drawn. Click again (or the ✕) to unpin. Multiple tokens can be pinned. Hover shows a transient preview.

Renderer: `src/goals/goalscope_viz.py`. Input = the `list[dict]` returned by `apply_goalscope(..., include_noncode=True, show=False)`.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
# resolve to the repo root so relative paths (head_sets/, notebooks/diff/visualizations/) work
# regardless of the notebook's launch cwd
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')
print('cwd:', os.getcwd())
from src.goals.goalscope_viz import goalscope_qualitative_html, show_goalscope_qualitative

cwd: /disk/u/arnab/Codes/Projects/goals


## GoalScope on Qwen2.5-Coder-32B
Load the model + the 122 goal heads, run `apply_goalscope` over a code prompt, and render. *(Heavy — needs a free GPU. Set `CUDA`.)*

In [2]:
import os, torch
MODEL = 'Qwen/Qwen2.5-Coder-32B-Instruct'
CUDA  = '7'   # set to free GPU(s)
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA
from src.models import ModelandTokenizer
from src.goals.goalscope import apply_goalscope
from scripts.evaluation.eval_goalscope import load_json_heads

slug = MODEL.split('/')[-1]
HEADS_JSON = f'head_sets/{slug}/goal__lam20_thr0.9_band25-56.json'  # 122 goal heads
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map='auto',
                       attn_implementation='eager')
heads = load_json_heads(HEADS_JSON, mt.n_layer, mt.model.config.num_attention_heads)
print(f'{len(heads)} goal heads loaded')

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

122 goal heads loaded


In [3]:
from src.functional import generate_with_patch
from rich import print as rich_print

task = "stable marriage"
# task = "merge-sort"

msgs = [
    {'role': 'system', 'content': 'You are a helpful assistant proficient in programming. '
        'Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.'},
    {'role': 'user', 'content': f'Implement the {task} algorithm in python'},
    {'role': 'assistant', 'content': '<code>\n'},
]
prompt = mt.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False,
                                             continue_final_message=True)

gen = generate_with_patch(
    mt=mt,
    inputs=prompt,
    n_gen_per_prompt=1,
    do_sample=False,
    remove_prefix=True,
    max_new_tokens=256
)[0]
rich_print(f"[white]{prompt}[/white][sea_green3]{gen}[/sea_green3]")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Implement the stable marriage algorithm in python<|im_end|>
<|im_start|>assistant
<code>
def stable_marriage(men_preferences, women_preferences):
    n = len(men_preferences)
    men_free = list(range(n))
    women_free = list(range(n))
    men_engaged_to = [None] * n
    women_engaged_to = [None] * n
    men_proposals = [0] * n

    while men_free:
        man = men_free.pop(0)
        woman = men_preferences[men_proposals]
        men_proposals += 1

        if women_engaged_to is None:
            women_engaged_to = man
            men_engaged_to = woman
        else:
            current_guy = women_engaged_to
            if women_preferences.index(man) < women_preferences.index(current_guy):
                women_engaged_to = man
                men_engaged_to = woman
                men_free.append(current_guy)
            else:
                men_free.append(man)

    return list(zip(men_engaged_to, women_engaged_to))

# Example usage:
men_prefs = [
    [1, 0, 2],
    [0, 

In [4]:
# Verbalize every code token (include_noncode=True keeps the whole prompt; non-code tokens dimmed).
res = apply_goalscope(prompt+gen, mt=mt, selected_heads=heads, strategy='additive_norm',
                      steer_layer_range=(35, 46), alpha=0.1, n_steer_tokens='all',
                      max_new_tokens=64, include_noncode=True, show=False)
print(sum(1 for r in res if r.get('scoped')), 'scoped tokens')

GoalScope[additive_norm]:   0%|          | 0/17 [00:00<?, ?it/s]

260 scoped tokens


In [ ]:
# Inline (interactive) + save a standalone .html for the paper.
import pathlib; pathlib.Path('notebooks/diff/visualizations').mkdir(parents=True, exist_ok=True)
goalscope_qualitative_html(res, title=f'{task} — GoalScope',
                           save_path=f'notebooks/diff/visualizations/{task}.html')
show_goalscope_qualitative(res, title=f'{task} — GoalScope', height=1024)

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


: 